# 06. Velocidade de Propagação dos Incêndios

Estima a velocidade de propagação de cada evento de incêndio a partir do deslocamento
do centróide diário de detecções MODIS consecutivas.

**Método:**
1. **DBSCAN espaço-temporal** (eps = 50 km, min_samples = 5) — agrupa focos próximos em eventos
2. **Divisão por lacuna temporal** (gap > 7 dias → novo sub-evento)
3. **Velocidade** = distância Haversine entre centróides diários consecutivos / Δt

**Entrada:** `data/inpe_queimadas/inpe_focos_2023_consolidado.csv`
**Saída:** `data/velocidade/focos_velocidade_2023.csv`

## 0. Dependências e configuração

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import unicodedata
from pathlib import Path

try:
    from sklearn.cluster import DBSCAN
    SKLEARN_OK = True
except ImportError:
    SKLEARN_OK = False
    print('[aviso] instale: pip install scikit-learn')

FOCOS_DIR = Path('data/inpe_queimadas')
OUT_DIR   = Path('data/velocidade')
OUT_DIR.mkdir(parents=True, exist_ok=True)

ANO      = 2023
MESES_PT = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
CORES_BIOMA = {
    'Amazonia'      : '#2ca02c', 'Cerrado'       : '#ff7f0e',
    'Caatinga'      : '#d62728', 'Mata Atlantica': '#9467bd',
    'Pantanal'      : '#17becf', 'Pampa'         : '#bcbd22',
}

def norm(s):
    return unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().strip()

print('Configuracao OK')

## 1. Carregamento dos focos

In [ ]:
df = pd.read_csv(
    FOCOS_DIR / f'inpe_focos_{ANO}_consolidado.csv',
    usecols=['latitude', 'longitude', 'data_pas', 'bioma', 'frp', 'mes', 'risco_fogo'],
    parse_dates=['data_pas'],
    low_memory=False,
)
df['data_pas'] = pd.to_datetime(df['data_pas']).dt.normalize()

print(f'Focos: {len(df):,}')
print(f'Periodo: {df["data_pas"].min().date()} -> {df["data_pas"].max().date()}')
df.head(3)

## 2. Clusterização DBSCAN (eventos de incêndio)

Usa métrica haversine para respeitar a curvatura da Terra.
Focos com label `-1` são ruído (detecções isoladas sem evento associado).

In [ ]:
out_cache = OUT_DIR / f'focos_velocidade_{ANO}.csv'

if out_cache.exists():
    df = pd.read_csv(out_cache, parse_dates=['data_pas'])
    print(f'[cache] {out_cache.name}  ({len(df):,} focos)')
    CACHE_LOADED = True
else:
    CACHE_LOADED = False

if not CACHE_LOADED and not SKLEARN_OK:
    print('[erro] scikit-learn necessario — instale e reexecute')

if not CACHE_LOADED and SKLEARN_OK:
    print('Clusterizando focos com DBSCAN (eps=50 km, min_samples=5)...')
    coords_rad = np.radians(df[['latitude', 'longitude']].values)
    eps_rad    = 50.0 / 6371.0  # 50 km em radianos

    db = DBSCAN(eps=eps_rad, min_samples=5,
                algorithm='ball_tree', metric='haversine', n_jobs=-1)
    df['cluster_id'] = db.fit_predict(coords_rad)

    n_clusters = (df['cluster_id'] >= 0).sum()
    n_noise    = (df['cluster_id'] == -1).sum()
    n_eventos  = df['cluster_id'].nunique() - 1
    print(f'Focos em eventos : {n_clusters:,} ({100*n_clusters/len(df):.1f}%)')
    print(f'Focos isolados   : {n_noise:,}   ({100*n_noise/len(df):.1f}%)')
    print(f'Clusters DBSCAN  : {n_eventos:,}')

    # Subdivide clusters por lacuna temporal > 7 dias
    MAX_GAP = 7
    df['evento_id'] = -1
    df_ev = df[df['cluster_id'] >= 0].copy().sort_values(['cluster_id', 'data_pas'])

    novo_id = 0
    for cid, grupo in df_ev.groupby('cluster_id'):
        datas = np.array(sorted(grupo['data_pas'].unique()))
        sub_ini = 0
        for i in range(1, len(datas)):
            gap = (pd.Timestamp(datas[i]) - pd.Timestamp(datas[i-1])).days
            if gap > MAX_GAP:
                mask = grupo['data_pas'].isin(datas[sub_ini:i])
                df.loc[grupo[mask].index, 'evento_id'] = novo_id
                novo_id += 1
                sub_ini = i
        mask = grupo['data_pas'].isin(datas[sub_ini:])
        df.loc[grupo[mask].index, 'evento_id'] = novo_id
        novo_id += 1

    n_sub = df['evento_id'].nunique() - 1
    print(f'Sub-eventos apos divisao temporal (gap>{MAX_GAP}d): {n_sub:,}')

## 3. Cálculo de velocidade de propagação

Para cada evento com ≥ 2 dias de detecção:
- Calcula centróide diário (média lat/lon)
- Mede distância Haversine entre centróides consecutivos
- Velocidade (m/s) = dist / Δt em segundos
- Descarta deslocamentos > 200 km/dia (artefato de associação errônea)

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


if not CACHE_LOADED and SKLEARN_OK:
    velocidades = []
    df_ev_ok = df[df['evento_id'] >= 0].copy()

    for eid, grupo in df_ev_ok.groupby('evento_id'):
        cents = (grupo.groupby('data_pas')[['latitude', 'longitude']]
                 .mean().reset_index().sort_values('data_pas'))
        if len(cents) < 2:
            continue
        for i in range(1, len(cents)):
            r0, r1 = cents.iloc[i-1], cents.iloc[i]
            dist_m = haversine_m(r0['latitude'], r0['longitude'],
                                 r1['latitude'], r1['longitude'])
            dt_s = (r1['data_pas'] - r0['data_pas']).total_seconds()
            if dt_s <= 0 or dist_m > 200_000:
                continue
            velocidades.append({
                'evento_id'     : eid,
                'velocidade_ms' : dist_m / dt_s,
                'velocidade_kmh': dist_m / dt_s * 3.6,
                'dist_km'       : dist_m / 1_000,
            })

    df_vel_ev = pd.DataFrame(velocidades)
    vel_media = (df_vel_ev.groupby('evento_id')['velocidade_ms']
                 .mean().reset_index())
    df = df.merge(vel_media, on='evento_id', how='left')
    df['velocidade_kmh'] = df['velocidade_ms'] * 3.6

    n_vel = df['velocidade_ms'].notna().sum()
    print(f'Focos com velocidade: {n_vel:,} ({100*n_vel/len(df):.1f}%)')
    print(f'Velocidade mediana  : {df["velocidade_ms"].median():.5f} m/s'
          f'  ({df["velocidade_ms"].median()*3.6:.4f} km/h)')
    print(f'Velocidade media    : {df["velocidade_ms"].mean():.5f} m/s')
    print(f'Velocidade maxima   : {df["velocidade_ms"].max():.5f} m/s')

    df.to_csv(out_cache, index=False)
    print(f'\n[ok] {out_cache}  ({out_cache.stat().st_size/1e6:.1f} MB)')

## 4. Estatísticas e visualizações

In [ ]:
if 'velocidade_ms' in df.columns:
    print('Velocidade de propagacao por bioma (m/s):')
    tab = (df[df['velocidade_ms'].notna()]
           .groupby('bioma')['velocidade_ms']
           .agg(['median', 'mean', 'std', 'count'])
           .round(5))
    print(tab.to_string())
    print()
    print('Percentis da velocidade (m/s):')
    print(df['velocidade_ms'].quantile([.25,.5,.75,.9,.95,.99]).round(5).to_string())

In [ ]:
if 'velocidade_ms' in df.columns:
    df_v = df[df['velocidade_ms'].notna() & (df['velocidade_ms'] > 0)]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'Velocidade de Propagacao dos Incendios — {ANO}',
                 fontsize=13, fontweight='bold')

    # 1. Histograma (escala log)
    ax = axes[0, 0]
    ax.hist(df_v['velocidade_ms'], bins=80, color='#ff7f0e', edgecolor='white', log=True)
    ax.axvline(df_v['velocidade_ms'].median(), color='black', linestyle='--', linewidth=1.2,
               label=f'Mediana: {df_v["velocidade_ms"].median():.5f} m/s')
    ax.set_xlabel('Velocidade (m/s)'); ax.set_ylabel('Frequencia (log)')
    ax.set_title('Distribuicao da Velocidade')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.25)

    # 2. Velocidade mediana por bioma
    ax = axes[0, 1]
    vel_bio = (df_v.groupby('bioma')['velocidade_ms']
               .median().sort_values())
    cores_b = [CORES_BIOMA.get(norm(b), '#aaaaaa') for b in vel_bio.index]
    ax.barh(vel_bio.index, vel_bio.values, color=cores_b, edgecolor='white')
    for i, v in enumerate(vel_bio.values):
        ax.text(v * 1.02, i, f'{v:.5f}', va='center', fontsize=7)
    ax.set_xlabel('Velocidade mediana (m/s)')
    ax.set_title('Velocidade Mediana por Bioma')
    ax.grid(axis='x', alpha=0.25)

    # 3. Velocidade mediana por mes
    ax = axes[1, 0]
    vel_mes = (df_v.groupby('mes')['velocidade_ms']
               .median().reindex(range(1, 13), fill_value=0))
    ax.bar(vel_mes.index, vel_mes.values, color='#d62728', edgecolor='white')
    ax.set_xticks(range(1, 13)); ax.set_xticklabels(MESES_PT, fontsize=8)
    ax.set_ylabel('Velocidade mediana (m/s)')
    ax.set_title('Sazonalidade da Velocidade')
    ax.grid(axis='y', alpha=0.25)

    # 4. Scatter FRP x Velocidade (amostra 5k)
    ax = axes[1, 1]
    smp = df_v.sample(min(5_000, len(df_v)), random_state=42)
    ax.scatter(smp['frp'], smp['velocidade_ms'], s=3, alpha=0.3,
               color='#1f77b4', rasterized=True)
    r = df_v[['frp', 'velocidade_ms']].dropna().corr().iloc[0, 1]
    ax.set_xlabel('FRP (MW)'); ax.set_ylabel('Velocidade (m/s)')
    ax.set_title(f'FRP x Velocidade  (r = {r:.3f})')
    ax.grid(True, alpha=0.25)

    plt.tight_layout()
    plt.savefig(OUT_DIR / f'fig_velocidade_{ANO}.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('[aviso] velocidade nao calculada — verifique etapas anteriores')

## 5. Exportação

In [ ]:
print(f'Arquivo: {out_cache}')
if out_cache.exists():
    print(f'Tamanho: {out_cache.stat().st_size/1e6:.1f} MB')
if 'evento_id' in df.columns:
    n_ev  = df.loc[df['evento_id'] >= 0, 'evento_id'].nunique()
    n_vel = df['velocidade_ms'].notna().sum() if 'velocidade_ms' in df.columns else 0
    print(f'Eventos detectados:           {n_ev:,}')
    print(f'Focos com velocidade estimada: {n_vel:,}')
print('Pronto para o notebook 07 (consolidacao).')